# EduManage Component Flow Demo
> Simple runnable notebook showing each platform component as a Python function.
>
> Every function accepts a request dictionary, makes decisions using if and else statements, and returns the request to the next component in the system.

## 0. Setup
This notebook uses only standard Python. It demonstrates system flow, not production implementation.

In [ ]:
from datetime import datetime
import uuid


def make_id(prefix):
    return f"{prefix}_{uuid.uuid4().hex[:6]}"


def log(request, message):
    request["trace"].append(message)
    return request


def make_request(action, user_role, payload):
    return {
        "request_id": make_id("req"),
        "action": action,
        "user_role": user_role,
        "payload": payload,
        "status": "new",
        "next_component": "internet_gateway",
        "trace": [],
        "response": None
    }


---
## 1. Public Entry Components
These functions simulate the public path of a request: internet gateway, CDN, WAF, load balancer, and API gateway.

In [ ]:
def internet_gateway(request):
    log(request, "Internet Gateway: received public request")
    request["next_component"] = "cdn"
    return request


def cdn(request):
    action = request["action"]
    if action == "view_resource":
        log(request, "CDN: static resource request can be served from cache")
        request["next_component"] = "resource_service"
    else:
        log(request, "CDN: dynamic request forwarded to WAF")
        request["next_component"] = "waf"
    return request


def waf(request):
    payload = request["payload"]
    if payload.get("blocked") is True:
        log(request, "WAF: request blocked because it looks unsafe")
        request["status"] = "rejected"
        request["next_component"] = "end"
        request["response"] = "Request rejected by WAF"
    else:
        log(request, "WAF: request allowed")
        request["next_component"] = "load_balancer"
    return request


def load_balancer(request):
    if request["request_id"][-1] in "02468":
        app_server = "app_server_a"
    else:
        app_server = "app_server_b"

    request["payload"]["app_server"] = app_server
    log(request, f"Load Balancer: routed request to {app_server}")
    request["next_component"] = "api_gateway"
    return request


def api_gateway(request):
    action = request["action"]
    allowed_actions = ["login", "create_class", "submit_assignment", "grade_assignment", "send_message", "view_report", "view_resource"]

    if action not in allowed_actions:
        log(request, "API Gateway: unknown action")
        request["status"] = "failed"
        request["next_component"] = "end"
        request["response"] = "Invalid API action"
    else:
        log(request, "API Gateway: action accepted and routed to authentication")
        request["next_component"] = "auth_service"
    return request


---
## 2. Security Components
Authentication checks identity. RBAC checks whether the user role can perform the requested action.

In [ ]:
def auth_service(request):
    if request["action"] == "login":
        log(request, "Auth Service: login request accepted")
        request["next_component"] = "notification_service"
    elif request["payload"].get("token") == "valid-token":
        log(request, "Auth Service: token is valid")
        request["next_component"] = "rbac_service"
    else:
        log(request, "Auth Service: token missing or invalid")
        request["status"] = "failed"
        request["next_component"] = "end"
        request["response"] = "Authentication failed"
    return request


def rbac_service(request):
    role = request["user_role"]
    action = request["action"]

    teacher_actions = ["create_class", "grade_assignment", "send_message", "view_report", "view_resource"]
    student_actions = ["submit_assignment", "send_message", "view_resource"]
    admin_actions = ["view_report", "view_resource"]

    if role == "teacher" and action in teacher_actions:
        log(request, "RBAC Service: teacher permission granted")
        request["next_component"] = "service_router"
    elif role == "student" and action in student_actions:
        log(request, "RBAC Service: student permission granted")
        request["next_component"] = "service_router"
    elif role == "admin" and action in admin_actions:
        log(request, "RBAC Service: admin permission granted")
        request["next_component"] = "service_router"
    else:
        log(request, "RBAC Service: permission denied")
        request["status"] = "failed"
        request["next_component"] = "end"
        request["response"] = "Access denied"
    return request


---
## 3. Service Router
This component decides which internal service should process the request next.

In [ ]:
def service_router(request):
    action = request["action"]

    if action == "create_class":
        request["next_component"] = "classroom_service"
    elif action == "submit_assignment":
        request["next_component"] = "assignment_service"
    elif action == "grade_assignment":
        request["next_component"] = "grading_service"
    elif action == "send_message":
        request["next_component"] = "communication_service"
    elif action == "view_report":
        request["next_component"] = "analytics_service"
    elif action == "view_resource":
        request["next_component"] = "resource_service"
    else:
        request["next_component"] = "end"
        request["status"] = "failed"
        request["response"] = "No matching service found"

    log(request, f"Service Router: next component is {request['next_component']}")
    return request


---
## 4. Core Platform Components
These functions represent EduManage business modules.

In [ ]:
DATABASE = {
    "classes": [],
    "assignments": [],
    "grades": [],
    "messages": [],
    "resources": ["chapter_1_notes.pdf", "python_lab_video.mp4"]
}

OBJECT_STORAGE = {}
MESSAGE_QUEUE = []


def classroom_service(request):
    title = request["payload"].get("class_title")
    if title:
        class_record = {"class_id": make_id("class"), "title": title, "created_at": str(datetime.now())}
        DATABASE["classes"].append(class_record)
        request["payload"]["class_id"] = class_record["class_id"]
        log(request, "Classroom Service: classroom created in SQL database")
        request["next_component"] = "notification_service"
    else:
        log(request, "Classroom Service: class title missing")
        request["status"] = "failed"
        request["next_component"] = "end"
        request["response"] = "Classroom creation failed"
    return request


def assignment_service(request):
    file_name = request["payload"].get("file_name")
    file_content = request["payload"].get("file_content")

    if file_name and file_content:
        file_key = f"submissions/{make_id('file')}_{file_name}"
        OBJECT_STORAGE[file_key] = file_content
        submission = {"submission_id": make_id("sub"), "file_key": file_key, "status": "submitted"}
        DATABASE["assignments"].append(submission)
        request["payload"]["submission_id"] = submission["submission_id"]
        log(request, "Assignment Service: file saved to object storage and submission saved to database")
        request["next_component"] = "message_queue"
    else:
        log(request, "Assignment Service: missing file name or file content")
        request["status"] = "failed"
        request["next_component"] = "end"
        request["response"] = "Submission failed"
    return request


def grading_service(request):
    score = request["payload"].get("score")
    max_score = request["payload"].get("max_score", 100)

    if score is None:
        log(request, "Grading Service: score missing")
        request["status"] = "failed"
        request["next_component"] = "end"
        request["response"] = "Grade failed"
    elif score < 0 or score > max_score:
        log(request, "Grading Service: score outside allowed range")
        request["status"] = "failed"
        request["next_component"] = "end"
        request["response"] = "Invalid score"
    else:
        grade = {"grade_id": make_id("grade"), "score": score, "max_score": max_score}
        DATABASE["grades"].append(grade)
        log(request, "Grading Service: grade saved to SQL database")
        request["next_component"] = "notification_service"
    return request


def communication_service(request):
    message = request["payload"].get("message")
    if message:
        DATABASE["messages"].append({"message_id": make_id("msg"), "message": message})
        log(request, "Communication Service: message saved to NoSQL communication log")
        request["next_component"] = "message_queue"
    else:
        log(request, "Communication Service: empty message rejected")
        request["status"] = "failed"
        request["next_component"] = "end"
        request["response"] = "Message failed"
    return request


def resource_service(request):
    resource_name = request["payload"].get("resource_name")
    if resource_name in DATABASE["resources"]:
        log(request, "Resource Service: resource found in object storage")
        request["status"] = "success"
        request["next_component"] = "end"
        request["response"] = f"Resource delivered: {resource_name}"
    else:
        log(request, "Resource Service: resource not found")
        request["status"] = "failed"
        request["next_component"] = "end"
        request["response"] = "Resource unavailable"
    return request


def analytics_service(request):
    if len(DATABASE["grades"]) == 0:
        average_score = 0
    else:
        total = sum(grade["score"] for grade in DATABASE["grades"])
        average_score = total / len(DATABASE["grades"])

    log(request, "Analytics Service: report generated from database records")
    request["status"] = "success"
    request["next_component"] = "end"
    request["response"] = {
        "class_count": len(DATABASE["classes"]),
        "submission_count": len(DATABASE["assignments"]),
        "grade_count": len(DATABASE["grades"]),
        "average_score": average_score
    }
    return request


---
## 5. Async and Notification Components
These functions simulate a message queue, background worker, and notification service.

In [ ]:
def message_queue(request):
    event = {"event_id": make_id("event"), "action": request["action"], "payload": request["payload"]}
    MESSAGE_QUEUE.append(event)

    if request["action"] == "submit_assignment":
        log(request, "Message Queue: submission event queued for teacher notification")
    elif request["action"] == "send_message":
        log(request, "Message Queue: message event queued for recipients")
    else:
        log(request, "Message Queue: event queued for background processing")

    request["next_component"] = "worker_service"
    return request


def worker_service(request):
    if len(MESSAGE_QUEUE) > 0:
        event = MESSAGE_QUEUE.pop(0)
        request["payload"]["processed_event"] = event["event_id"]
        log(request, "Worker Service: event processed successfully")
        request["next_component"] = "notification_service"
    else:
        log(request, "Worker Service: no queued event found")
        request["next_component"] = "notification_service"
    return request


def notification_service(request):
    action = request["action"]

    if action == "login":
        message = "Login successful"
    elif action == "create_class":
        message = "Classroom created and students can be invited"
    elif action == "submit_assignment":
        message = "Assignment submitted and teacher notified"
    elif action == "grade_assignment":
        message = "Grade saved and student notified"
    elif action == "send_message":
        message = "Message delivered to classroom members"
    else:
        message = "Request completed"

    log(request, f"Notification Service: {message}")
    request["status"] = "success"
    request["next_component"] = "end"
    request["response"] = message
    return request


---
## 6. Workflow Engine
The workflow engine repeatedly sends the request to the component named in `next_component` until the request reaches `end`.

In [ ]:
COMPONENTS = {
    "internet_gateway": internet_gateway,
    "cdn": cdn,
    "waf": waf,
    "load_balancer": load_balancer,
    "api_gateway": api_gateway,
    "auth_service": auth_service,
    "rbac_service": rbac_service,
    "service_router": service_router,
    "classroom_service": classroom_service,
    "assignment_service": assignment_service,
    "grading_service": grading_service,
    "communication_service": communication_service,
    "resource_service": resource_service,
    "analytics_service": analytics_service,
    "message_queue": message_queue,
    "worker_service": worker_service,
    "notification_service": notification_service
}


def run_workflow(request):
    while request["next_component"] != "end":
        component_name = request["next_component"]
        component = COMPONENTS[component_name]
        request = component(request)
    return request


def print_result(request):
    print("=" * 60)
    print(f"Request: {request['request_id']}")
    print(f"Action: {request['action']}")
    print(f"Role: {request['user_role']}")
    print(f"Status: {request['status']}")
    print(f"Response: {request['response']}")
    print("Trace:")
    for step in request["trace"]:
        print(f"- {step}")


---
## 7. Demo 1: Teacher Creates a Classroom
The request passes through public access, security checks, service routing, classroom service, and notification service.

In [ ]:
request = make_request(
    action="create_class",
    user_role="teacher",
    payload={"token": "valid-token", "class_title": "Class 10 - Computer Science"}
)

result = run_workflow(request)
print_result(result)


---
## 8. Demo 2: Student Submits an Assignment
This request reaches assignment service, object storage, database, message queue, worker service, and notification service.

In [ ]:
request = make_request(
    action="submit_assignment",
    user_role="student",
    payload={
        "token": "valid-token",
        "file_name": "functions_lab.py",
        "file_content": "def add(a, b): return a + b"
    }
)

result = run_workflow(request)
print_result(result)


---
## 9. Demo 3: Teacher Grades an Assignment
This request shows validation using if and else statements inside the grading service.

In [ ]:
request = make_request(
    action="grade_assignment",
    user_role="teacher",
    payload={"token": "valid-token", "score": 91, "max_score": 100}
)

result = run_workflow(request)
print_result(result)


---
## 10. Demo 4: Invalid Role Decision
A student tries to grade an assignment. RBAC uses if and else checks to stop the request.

In [ ]:
request = make_request(
    action="grade_assignment",
    user_role="student",
    payload={"token": "valid-token", "score": 80, "max_score": 100}
)

result = run_workflow(request)
print_result(result)


---
## 11. Demo 5: Analytics Report
The admin can request a report. The analytics service reads data collected by earlier workflows.

In [ ]:
request = make_request(
    action="view_report",
    user_role="admin",
    payload={"token": "valid-token"}
)

result = run_workflow(request)
print_result(result)


---
## Summary

- Each architecture component is represented as a function.
- Every function receives the request, uses if and else decisions, and sets `next_component`.
- The workflow engine moves the request through the system until it reaches `end`.
- The trace output shows how data travels across EduManage components.